# 500 — CRISPR Associations

This notebook evaluates associations between the frozen Phase 4 consensus transcriptomic programs and DepMap Public 24Q4 CRISPR gene-effect profiles in cancer cell models with both measurements.

The primary screen adjusts for cell-line lineage. Within-lineage estimates and pooled correlations provide descriptive context for interpreting the primary associations.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json
import re

import numpy as np
import pandas as pd
import statsmodels.api as sm

from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

from pancancer_epigenetics.analysis_parameters import (
    align_program_columns,
    load_consensus_programs,
)
from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Input paths
# =============================================================================

crispr_gene_effect_path = Paths.depmap / "CRISPRGeneEffect.csv"

consensus_cellline_scores_path = (
    Paths.consensus_programs
    / "401_consensus_cellline_scores.parquet"
)

cell_line_cohort_path = (
    Paths.metadata
    / "302_integrated_modeling_cohort.csv"
)

In [3]:
# =============================================================================
# Load direct inputs
# =============================================================================

consensus_program_ids = [program.program_id for program in load_consensus_programs()]
consensus_cellline_scores = align_program_columns(
    pd.read_parquet(consensus_cellline_scores_path),
    consensus_program_ids,
    metadata_columns=("ModelID",),
)

crispr_gene_effect = pd.read_csv(crispr_gene_effect_path)

cell_line_cohort = pd.read_csv(cell_line_cohort_path)

In [4]:
# =============================================================================
# Inspect loaded input structure
# =============================================================================

print("Consensus cell-line scores:", consensus_cellline_scores.shape)
print("Consensus columns:", consensus_cellline_scores.columns.tolist())

print("\nCRISPR gene effect:", crispr_gene_effect.shape)
print("CRISPR first columns:", crispr_gene_effect.columns[:5].tolist())

Consensus cell-line scores: (712, 3)
Consensus columns: ['ModelID', 'CONSENSUS_TX_01', 'CONSENSUS_TX_02']

CRISPR gene effect: (1178, 17917)
CRISPR first columns: ['Unnamed: 0', 'A1BG (1)', 'A1CF (29974)', 'A2M (2)', 'A2ML1 (144568)']


In [5]:
# =============================================================================
# Standardize CRISPR model identifier
# =============================================================================

crispr_gene_effect = crispr_gene_effect.rename(
    columns={"Unnamed: 0": "ModelID"}
)

In [6]:
# =============================================================================
# Characterize model overlap
# =============================================================================

for source, model_ids in {
    "Notebook 401 scores": consensus_cellline_scores["ModelID"],
    "Notebook 302 cohort": cell_line_cohort["ModelID"],
    "CRISPR gene effect": crispr_gene_effect["ModelID"],
}.items():
    if model_ids.isna().any() or not model_ids.is_unique:
        raise ValueError(f"{source} must have non-missing, unique ModelIDs")

consensus_model_ids = set(consensus_cellline_scores["ModelID"])
crispr_model_ids = set(crispr_gene_effect["ModelID"])

cohort_model_ids = set(cell_line_cohort["ModelID"])
if consensus_model_ids != cohort_model_ids:
    raise ValueError(
        "Notebook 401 score ModelIDs differ from the Notebook 302 cohort: "
        f"missing={sorted(cohort_model_ids - consensus_model_ids)}, "
        f"unexpected={sorted(consensus_model_ids - cohort_model_ids)}"
    )

shared_model_ids = consensus_model_ids & crispr_model_ids

print("Frozen consensus models:", len(consensus_model_ids))
print("CRISPR models:", len(crispr_model_ids))
print("Shared models:", len(shared_model_ids))
print(
    "Frozen consensus models without CRISPR:",
    len(consensus_model_ids - crispr_model_ids),
)

Frozen consensus models: 712
CRISPR models: 1178
Shared models: 538
Frozen consensus models without CRISPR: 174


In [7]:
# =============================================================================
# Align inputs to the shared model cohort
# =============================================================================

shared_consensus_scores = (
    consensus_cellline_scores[
        consensus_cellline_scores["ModelID"].isin(crispr_model_ids)
    ]
    .set_index("ModelID")
    .copy()
)

shared_crispr_gene_effect = (
    crispr_gene_effect
    .set_index("ModelID")
    .loc[shared_consensus_scores.index]
    .copy()
)

In [8]:
# =============================================================================
# Parse CRISPR gene identifiers
# =============================================================================

crispr_gene_annotations = pd.DataFrame(
    {"source_gene_label": shared_crispr_gene_effect.columns}
)

parsed_gene_ids = crispr_gene_annotations["source_gene_label"].str.extract(
    r"^(?P<gene_symbol>.+) \((?P<entrez_id>\d+)\)$"
)

crispr_gene_annotations = pd.concat(
    [crispr_gene_annotations, parsed_gene_ids],
    axis=1,
)

crispr_gene_annotations["entrez_id"] = (
    pd.to_numeric(
        crispr_gene_annotations["entrez_id"],
        errors="coerce",
    )
    .astype("Int64")
)

In [9]:
# =============================================================================
# Summarize CRISPR gene identifier parsing
# =============================================================================

print("CRISPR gene labels:", len(crispr_gene_annotations))
print(
    "Unparsed labels:",
    crispr_gene_annotations["gene_symbol"].isna().sum(),
)
print(
    "Duplicate source labels:",
    crispr_gene_annotations["source_gene_label"].duplicated().sum(),
)
print(
    "Gene symbols with multiple source labels:",
    crispr_gene_annotations["gene_symbol"].duplicated(keep=False).sum(),
)
print(
    "Entrez IDs with multiple source labels:",
    crispr_gene_annotations["entrez_id"].duplicated(keep=False).sum(),
)

CRISPR gene labels: 17916
Unparsed labels: 0
Duplicate source labels: 0
Gene symbols with multiple source labels: 0
Entrez IDs with multiple source labels: 0


In [10]:
# =============================================================================
# Characterize CRISPR gene coverage
# =============================================================================

crispr_gene_coverage = crispr_gene_annotations.copy()

crispr_gene_coverage["n_observed"] = (
    shared_crispr_gene_effect.notna().sum(axis=0).to_numpy()
)

crispr_gene_coverage["n_missing"] = (
    shared_crispr_gene_effect.isna().sum(axis=0).to_numpy()
)

crispr_gene_coverage["missing_fraction"] = (
    crispr_gene_coverage["n_missing"]
    / len(shared_crispr_gene_effect)
)

In [11]:
# =============================================================================
# Summarize CRISPR gene coverage
# =============================================================================

print(
    crispr_gene_coverage[
        ["n_observed", "n_missing", "missing_fraction"]
    ].describe()
)

print(
    "\nGenes complete across shared models:",
    (crispr_gene_coverage["n_missing"] == 0).sum(),
)

         n_observed     n_missing  missing_fraction
count  17916.000000  17916.000000      17916.000000
mean     534.135745      3.864255          0.007183
std       25.291744     25.291744          0.047011
min      262.000000      0.000000          0.000000
25%      538.000000      0.000000          0.000000
50%      538.000000      0.000000          0.000000
75%      538.000000      0.000000          0.000000
max      538.000000    276.000000          0.513011

Genes complete across shared models: 17097


In [12]:
# =============================================================================
# Characterize CRISPR gene informativeness
# =============================================================================

crispr_gene_coverage["gene_effect_sd"] = (
    shared_crispr_gene_effect.std(axis=0).to_numpy()
)

crispr_gene_coverage["n_unique"] = (
    shared_crispr_gene_effect.nunique(axis=0).to_numpy()
)

In [13]:
# =============================================================================
# Summarize CRISPR gene informativeness
# =============================================================================

print(
    crispr_gene_coverage[
        ["gene_effect_sd", "n_unique"]
    ].describe()
)

print(
    "\nGenes with zero variance:",
    (crispr_gene_coverage["gene_effect_sd"] == 0).sum(),
)
print(
    "Genes with fewer than 2 observed values:",
    (crispr_gene_coverage["n_observed"] < 2).sum(),
)

       gene_effect_sd      n_unique
count    17916.000000  17916.000000
mean         0.141347    534.135745
std          0.071635     25.291744
min          0.061045    262.000000
25%          0.102309    538.000000
50%          0.115769    538.000000
75%          0.142916    538.000000
max          0.853411    538.000000

Genes with zero variance: 0
Genes with fewer than 2 observed values: 0


In [14]:
# =============================================================================
# Freeze CRISPR gene eligibility
# =============================================================================

MIN_CRISPR_MODEL_FRACTION = 0.90
MIN_CRISPR_MODELS = int(
    np.ceil(MIN_CRISPR_MODEL_FRACTION * len(shared_crispr_gene_effect))
)

crispr_gene_coverage["analysis_eligible"] = (
    (crispr_gene_coverage["n_observed"] >= MIN_CRISPR_MODELS)
    & (crispr_gene_coverage["n_unique"] >= 2)
    & (crispr_gene_coverage["gene_effect_sd"] > 0)
)

In [15]:
# =============================================================================
# Summarize frozen CRISPR gene universe
# =============================================================================

eligible_gene_count = crispr_gene_coverage["analysis_eligible"].sum()

print("Minimum CRISPR models required:", MIN_CRISPR_MODELS)
print("Eligible CRISPR genes:", eligible_gene_count)
print(
    "Excluded CRISPR genes:",
    len(crispr_gene_coverage) - eligible_gene_count,
)

Minimum CRISPR models required: 485
Eligible CRISPR genes: 17205
Excluded CRISPR genes: 711


## CRISPR cohort and gene eligibility

**Saved execution.** Of the 712 consensus cell models, 538 have CRISPR gene-effect data. The analysis retains the original DepMap `SYMBOL (EntrezID)` labels as gene identifiers.

A gene is eligible if its effect is observed in at least 90% of the shared cohort (485 models) and has nonzero variance. This rule retained 17,205 of 17,916 genes.

In [16]:
# =============================================================================
# Construct lineage-aware shared cohort
# =============================================================================

shared_consensus_analysis = (
    shared_consensus_scores
    .reset_index()
    .merge(
        cell_line_cohort[
            [
                "ModelID",
                "OncotreeLineage",
            ]
        ],
        on="ModelID",
        how="left",
        validate="one_to_one",
    )
    .set_index("ModelID")
)

In [17]:
# =============================================================================
# Characterize lineage support in the shared cohort
# =============================================================================

MIN_LINEAGE_N = 15    # Freeze lineage-support parameter

shared_lineage_counts = (
    shared_consensus_analysis["OncotreeLineage"]
    .value_counts()
    .rename_axis("OncotreeLineage")
    .reset_index(name="n_models")
)

shared_lineage_counts["lineage_supported"] = (
    shared_lineage_counts["n_models"] >= MIN_LINEAGE_N
)

print("Shared lineages:", len(shared_lineage_counts))
print(
    "Lineages with n >= MIN_LINEAGE_N:",
    shared_lineage_counts["lineage_supported"].sum(),
)
print(
    "Models in supported lineages:",
    shared_lineage_counts.loc[
        shared_lineage_counts["lineage_supported"],
        "n_models",
    ].sum(),
)

Shared lineages: 26
Lineages with n >= MIN_LINEAGE_N: 13
Models in supported lineages: 438


In [18]:
# =============================================================================
# Freeze primary analysis universes
# =============================================================================

eligible_crispr_genes = crispr_gene_coverage.loc[
    crispr_gene_coverage["analysis_eligible"],
    "source_gene_label",
].tolist()

supported_lineages = shared_lineage_counts.loc[
    shared_lineage_counts["lineage_supported"],
    "OncotreeLineage",
].tolist()

In [19]:
# =============================================================================
# Confirm lineage annotation coverage
# =============================================================================

missing_lineage_count = (
    shared_consensus_analysis["OncotreeLineage"]
    .isna()
    .sum()
)

print(
    "Shared models without lineage annotation:",
    missing_lineage_count,
)

Shared models without lineage annotation: 0


In [20]:
# =============================================================================
# Freeze primary CRISPR association parameters
# =============================================================================

PRIMARY_MIN_LINEAGE_N = 2

PRIMARY_FDR_ALPHA = 0.05
PRIMARY_FDR_METHOD = "fdr_bh"
PRIMARY_COV_TYPE = "HC3"

PRIMARY_HYPOTHESIS_COUNT = (
    len(consensus_program_ids)
    * len(eligible_crispr_genes)
)

# Lineages represented by a single observed model for a given gene do not
# contribute within-lineage information to the primary program coefficient.
# This estimability rule is distinct from MIN_LINEAGE_N = 15, which applies
# only to secondary within-lineage characterization.

# Primary model:
# gene effect ~ consensus score + lineage fixed effects
#
# Multiplicity is controlled globally across the full frozen
# program × eligible-gene hypothesis family.
#
# Pooled Spearman associations are descriptive only.

## Primary association model and lineage support

The primary model regresses CRISPR gene effect on the consensus-program score with lineage fixed effects and HC3 standard errors. Each fit retains lineages with at least two models with observed gene effect and program score. Benjamini–Hochberg correction is applied jointly across the complete program–gene test family at 5% FDR.

**Saved execution.** The family contained 34,410 tests. The shared cohort spans 26 lineages. For secondary within-lineage estimates, the minimum is 15 observed models per association. Thirteen lineages comprising 438 models meet the cohort-level support threshold.

In [21]:
# =============================================================================
# Define primary lineage-adjusted association model
# =============================================================================

def fit_lineage_adjusted_association(
    gene_effect,
    program_score,
):
    observed = gene_effect.notna() & program_score.notna()

    observed_lineages = shared_consensus_analysis.loc[
        observed,
        "OncotreeLineage",
    ]

    lineage_counts = observed_lineages.value_counts()

    retained_lineages = lineage_counts.loc[
        lineage_counts >= PRIMARY_MIN_LINEAGE_N
    ].index

    retained = (
        observed
        & shared_consensus_analysis["OncotreeLineage"].isin(
            retained_lineages
        )
    )

    lineage_design_local = pd.get_dummies(
        shared_consensus_analysis.loc[
            retained,
            "OncotreeLineage",
        ],
        prefix="lineage",
        drop_first=True,
        dtype=float,
    )

    design = pd.concat(
        [
            program_score.loc[retained].rename("program_score"),
            lineage_design_local,
        ],
        axis=1,
    )

    design = sm.add_constant(
        design,
        has_constant="add",
    )

    model = sm.OLS(
        gene_effect.loc[retained],
        design,
    ).fit(cov_type=PRIMARY_COV_TYPE)

    return {
        "n_models": int(retained.sum()),
        "beta": model.params["program_score"],
        "standard_error": model.bse["program_score"],
        "p_value": model.pvalues["program_score"],
    }

In [22]:
# =============================================================================
# Construct primary CRISPR analysis matrix
# =============================================================================

primary_crispr_gene_effect = (
    shared_crispr_gene_effect[
        eligible_crispr_genes
    ]
    .copy()
)

In [23]:
# =============================================================================
# Run primary lineage-adjusted CRISPR association screen
# =============================================================================

primary_association_records = []

for program_id in consensus_program_ids:
    program_score = shared_consensus_analysis[program_id]

    for gene_label in eligible_crispr_genes:
        gene_effect = primary_crispr_gene_effect[gene_label]

        association = fit_lineage_adjusted_association(
            gene_effect=gene_effect,
            program_score=program_score,
        )

        primary_association_records.append(
            {
                "consensus_program_id": program_id,
                "source_gene_label": gene_label,
                **association,
            }
        )

primary_crispr_associations = pd.DataFrame(
    primary_association_records
)

In [24]:
# =============================================================================
# Verify primary CRISPR association numerical stability
# =============================================================================

print(
    "Primary associations:",
    len(primary_crispr_associations),
)
print(
    "Non-finite betas:",
    (~np.isfinite(primary_crispr_associations["beta"])).sum(),
)
print(
    "Non-finite standard errors:",
    (~np.isfinite(
        primary_crispr_associations["standard_error"]
    )).sum(),
)
print(
    "Non-finite p-values:",
    (~np.isfinite(primary_crispr_associations["p_value"])).sum(),
)

Primary associations: 34410
Non-finite betas: 0
Non-finite standard errors: 0
Non-finite p-values: 0


In [25]:
# =============================================================================
# Apply global primary multiple-testing correction
# =============================================================================

primary_fdr_reject, primary_fdr_q_values, _, _ = multipletests(
    primary_crispr_associations["p_value"],
    alpha=PRIMARY_FDR_ALPHA,
    method=PRIMARY_FDR_METHOD,
)

primary_crispr_associations["fdr_q_value"] = primary_fdr_q_values
primary_crispr_associations["fdr_significant"] = primary_fdr_reject

In [26]:
# =============================================================================
# Summarize primary multiple-testing results
# =============================================================================

primary_fdr_summary = (
    primary_crispr_associations
    .groupby("consensus_program_id")
    .agg(
        n_tests=("source_gene_label", "size"),
        n_fdr_significant=("fdr_significant", "sum"),
        min_fdr_q_value=("fdr_q_value", "min"),
    )
    .reset_index()
)

print(
    "Primary FDR-significant associations:",
    primary_crispr_associations["fdr_significant"].sum(),
    "of",
    len(primary_crispr_associations),
)

primary_fdr_summary

Primary FDR-significant associations: 560 of 34410


,consensus_program_id,n_tests,n_fdr_significant,min_fdr_q_value
0,CONSENSUS_TX_01,17205,307,3.636153e-08
1,CONSENSUS_TX_02,17205,253,2.017552e-16


In [27]:
# =============================================================================
# Prepare FDR-significant primary associations
# =============================================================================

significant_primary_associations = (
    primary_crispr_associations.loc[
        primary_crispr_associations["fdr_significant"]
    ]
    .copy()
)

significant_primary_associations["effect_direction"] = np.where(
    significant_primary_associations["beta"] < 0,
    "higher_program_stronger_dependency",
    "higher_program_weaker_dependency",
)

In [28]:
# =============================================================================
# Summarize significant primary effect magnitudes
# =============================================================================

significant_primary_associations["absolute_beta"] = (
    significant_primary_associations["beta"].abs()
)

primary_effect_summary = (
    significant_primary_associations
    .groupby(
        [
            "consensus_program_id",
            "effect_direction",
        ]
    )
    .agg(
        n_associations=("source_gene_label", "size"),
        median_beta=("beta", "median"),
        median_absolute_beta=("absolute_beta", "median"),
        max_absolute_beta=("absolute_beta", "max"),
    )
    .reset_index()
)

primary_effect_summary

,consensus_program_id,effect_direction,n_associations,median_beta,median_absolute_beta,max_absolute_beta
0,CONSENSUS_TX_01,higher_program_stronger_dependency,226,-0.144598,0.144598,0.337719
1,CONSENSUS_TX_01,higher_program_weaker_dependency,81,0.121654,0.121654,0.439680
2,CONSENSUS_TX_02,higher_program_stronger_dependency,115,-0.049984,0.049984,0.209848
3,CONSENSUS_TX_02,higher_program_weaker_dependency,138,0.057365,0.057365,0.230221


## Primary CRISPR associations

**Saved execution.** The primary screen identified 560 FDR-significant program–gene associations: 307 for `CONSENSUS_TX_01` and 253 for `CONSENSUS_TX_02`. A negative coefficient means that higher program scores are associated with more negative CRISPR gene effect, consistent with stronger dependency.

In [29]:
# =============================================================================
# Define within-lineage association estimator
# =============================================================================

def estimate_within_lineage_association(
    gene_effect,
    program_score,
    lineage,
):
    lineage_mask = (
        shared_consensus_analysis["OncotreeLineage"]
        == lineage
    )

    observed = (
        lineage_mask
        & gene_effect.notna()
        & program_score.notna()
    )

    x = program_score.loc[observed]
    y = gene_effect.loc[observed]

    if len(x) < MIN_LINEAGE_N:
        return None

    x_centered = x - x.mean()
    denominator = np.sum(x_centered**2)

    if denominator == 0:
        return None

    beta = np.sum(
        x_centered * (y - y.mean())
    ) / denominator

    return {
        "OncotreeLineage": lineage,
        "n_models": len(x),
        "beta": beta,
    }

In [30]:
# =============================================================================
# Estimate within-lineage effects for primary significant associations
# =============================================================================

within_lineage_records = []

for row in significant_primary_associations.itertuples(index=False):
    program_score = shared_consensus_analysis[
        row.consensus_program_id
    ]
    gene_effect = primary_crispr_gene_effect[
        row.source_gene_label
    ]

    for lineage in supported_lineages:
        estimate = estimate_within_lineage_association(
            gene_effect=gene_effect,
            program_score=program_score,
            lineage=lineage,
        )

        if estimate is not None:
            within_lineage_records.append(
                {
                    "consensus_program_id": row.consensus_program_id,
                    "source_gene_label": row.source_gene_label,
                    **estimate,
                }
            )

within_lineage_associations = pd.DataFrame(
    within_lineage_records
)

In [31]:
# =============================================================================
# Summarize cross-lineage effect consistency
# =============================================================================

primary_effect_lookup = (
    significant_primary_associations[
        [
            "consensus_program_id",
            "source_gene_label",
            "beta",
        ]
    ]
    .rename(columns={"beta": "primary_beta"})
)

within_lineage_summary = (
    within_lineage_associations
    .merge(
        primary_effect_lookup,
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="many_to_one",
    )
)

within_lineage_summary["direction_consistent"] = (
    np.sign(within_lineage_summary["beta"])
    == np.sign(within_lineage_summary["primary_beta"])
)

cross_lineage_consistency = (
    within_lineage_summary
    .groupby(
        [
            "consensus_program_id",
            "source_gene_label",
        ]
    )
    .agg(
        n_evaluable_lineages=("OncotreeLineage", "nunique"),
        n_direction_consistent=("direction_consistent", "sum"),
        median_lineage_beta=("beta", "median"),
        min_lineage_beta=("beta", "min"),
        max_lineage_beta=("beta", "max"),
    )
    .reset_index()
)

cross_lineage_consistency["direction_consistency_fraction"] = (
    cross_lineage_consistency["n_direction_consistent"]
    / cross_lineage_consistency["n_evaluable_lineages"]
)

In [32]:
# =============================================================================
# Summarize cross-lineage consistency distribution
# =============================================================================

cross_lineage_consistency = (
    cross_lineage_consistency
    .merge(
        significant_primary_associations[
            [
                "consensus_program_id",
                "source_gene_label",
                "effect_direction",
            ]
        ],
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

cross_lineage_consistency_summary = (
    cross_lineage_consistency
    .groupby(
        [
            "consensus_program_id",
            "effect_direction",
        ]
    )
    .agg(
        n_associations=("source_gene_label", "size"),
        median_evaluable_lineages=("n_evaluable_lineages", "median"),
        min_evaluable_lineages=("n_evaluable_lineages", "min"),
        median_direction_consistency=(
            "direction_consistency_fraction",
            "median",
        ),
        fully_direction_consistent=(
            "direction_consistency_fraction",
            lambda x: (x == 1).sum(),
        ),
    )
    .reset_index()
)

cross_lineage_consistency_summary

,consensus_program_id,effect_direction,n_associations,median_evaluable_lineages,min_evaluable_lineages,median_direction_consistency,fully_direction_consistent
0,CONSENSUS_TX_01,higher_program_stronger_dependency,226,13.0,13,0.807692,13
1,CONSENSUS_TX_01,higher_program_weaker_dependency,81,13.0,13,0.846154,4
2,CONSENSUS_TX_02,higher_program_stronger_dependency,115,13.0,13,0.692308,1
3,CONSENSUS_TX_02,higher_program_weaker_dependency,138,13.0,13,0.769231,2


In [33]:
# =============================================================================
# Quantify cross-lineage effect heterogeneity
# =============================================================================

cross_lineage_heterogeneity = (
    within_lineage_associations
    .groupby(
        [
            "consensus_program_id",
            "source_gene_label",
        ]
    )
    .agg(
        lineage_beta_sd=("beta", "std"),
        lineage_beta_q25=("beta", lambda x: x.quantile(0.25)),
        lineage_beta_q75=("beta", lambda x: x.quantile(0.75)),
        lineage_beta_min=("beta", "min"),
        lineage_beta_max=("beta", "max"),
    )
    .reset_index()
)

cross_lineage_heterogeneity["lineage_beta_iqr"] = (
    cross_lineage_heterogeneity["lineage_beta_q75"]
    - cross_lineage_heterogeneity["lineage_beta_q25"]
)

cross_lineage_heterogeneity["lineage_beta_range"] = (
    cross_lineage_heterogeneity["lineage_beta_max"]
    - cross_lineage_heterogeneity["lineage_beta_min"]
)

In [34]:
# =============================================================================
# Summarize cross-lineage effect heterogeneity
# =============================================================================

cross_lineage_heterogeneity = (
    cross_lineage_heterogeneity
    .merge(
        significant_primary_associations[
            [
                "consensus_program_id",
                "source_gene_label",
                "effect_direction",
            ]
        ],
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

cross_lineage_heterogeneity_summary = (
    cross_lineage_heterogeneity
    .groupby(
        [
            "consensus_program_id",
            "effect_direction",
        ]
    )
    .agg(
        n_associations=("source_gene_label", "size"),
        median_lineage_beta_sd=("lineage_beta_sd", "median"),
        median_lineage_beta_iqr=("lineage_beta_iqr", "median"),
        median_lineage_beta_range=("lineage_beta_range", "median"),
    )
    .reset_index()
)

cross_lineage_heterogeneity_summary

,consensus_program_id,effect_direction,n_associations,median_lineage_beta_sd,median_lineage_beta_iqr,median_lineage_beta_range
0,CONSENSUS_TX_01,higher_program_stronger_dependency,226,0.221930,0.229756,0.772690
1,CONSENSUS_TX_01,higher_program_weaker_dependency,81,0.224069,0.201826,0.804054
2,CONSENSUS_TX_02,higher_program_stronger_dependency,115,0.137987,0.091832,0.501501
3,CONSENSUS_TX_02,higher_program_weaker_dependency,138,0.143829,0.097955,0.523830


## Within-lineage consistency and heterogeneity

**Saved execution.** All 560 significant associations were evaluable in 13 lineages. Among stronger-dependency associations, median directional agreement with the primary coefficient was 80.8% for `CONSENSUS_TX_01` and 69.2% for `CONSENSUS_TX_02`. Within-lineage effect magnitudes also varied, with greater median dispersion for `CONSENSUS_TX_01`.

In [35]:
# =============================================================================
# Integrate primary and cross-lineage association evidence
# =============================================================================

primary_significant_evidence = (
    significant_primary_associations
    .merge(
        cross_lineage_consistency[
            [
                "consensus_program_id",
                "source_gene_label",
                "n_evaluable_lineages",
                "n_direction_consistent",
                "direction_consistency_fraction",
                "median_lineage_beta",
            ]
        ],
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        cross_lineage_heterogeneity[
            [
                "consensus_program_id",
                "source_gene_label",
                "lineage_beta_sd",
                "lineage_beta_iqr",
                "lineage_beta_range",
            ]
        ],
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

In [36]:
# =============================================================================
# Add CRISPR gene annotations to integrated evidence
# =============================================================================

primary_significant_evidence = (
    primary_significant_evidence
    .merge(
        crispr_gene_annotations[
            [
                "source_gene_label",
                "gene_symbol",
                "entrez_id",
            ]
        ],
        on="source_gene_label",
        how="left",
        validate="many_to_one",
    )
)

In [37]:
# =============================================================================
# Characterize gene-level overlap across consensus programs
# =============================================================================

significant_gene_program_count = primary_significant_evidence.groupby(
    "source_gene_label"
)["consensus_program_id"].nunique()

print(
    "Unique genes with primary FDR-significant associations:",
    significant_gene_program_count.size,
)

for n_programs in range(1, len(consensus_program_ids) + 1):
    print(
        f"Genes associated with exactly {n_programs} program(s):",
        significant_gene_program_count.eq(n_programs).sum(),
    )

Unique genes with primary FDR-significant associations: 532
Genes associated with one program: 504
Genes associated with two programs: 28


In [38]:
# =============================================================================
# Characterize effect direction across multi-program genes
# =============================================================================

multi_program_gene_direction = (
    primary_significant_evidence
    .groupby("source_gene_label")
    .agg(
        n_programs=("consensus_program_id", "nunique"),
        n_effect_directions=("effect_direction", "nunique"),
    )
    .reset_index()
)

multi_program_gene_direction = (
    multi_program_gene_direction.loc[
        multi_program_gene_direction["n_programs"] > 1
    ]
    .copy()
)

multi_program_gene_direction["direction_consistent_across_programs"] = (
    multi_program_gene_direction["n_effect_directions"] == 1
)

print("Multi-program genes:", len(multi_program_gene_direction))
print(
    "Same direction across programs:",
    multi_program_gene_direction[
        "direction_consistent_across_programs"
    ].sum(),
)
print(
    "Opposite directions across programs:",
    (
        ~multi_program_gene_direction[
            "direction_consistent_across_programs"
        ]
    ).sum(),
)

Multi-program genes: 28
Same direction across programs: 27
Opposite directions across programs: 1


In [39]:
# =============================================================================
# Calculate descriptive pooled associations
# =============================================================================

pooled_association_records = []

for row in primary_significant_evidence.itertuples(index=False):
    gene_effect = primary_crispr_gene_effect[
        row.source_gene_label
    ]
    program_score = shared_consensus_analysis[
        row.consensus_program_id
    ]

    observed = gene_effect.notna() & program_score.notna()

    pooled_spearman_rho, _ = spearmanr(
        program_score.loc[observed],
        gene_effect.loc[observed],
    )

    pooled_association_records.append(
        {
            "consensus_program_id": row.consensus_program_id,
            "source_gene_label": row.source_gene_label,
            "pooled_spearman_rho": pooled_spearman_rho,
        }
    )

pooled_primary_associations = pd.DataFrame(
    pooled_association_records
)

In [40]:
# =============================================================================
# Add descriptive pooled associations to integrated evidence
# =============================================================================

primary_significant_evidence = (
    primary_significant_evidence
    .merge(
        pooled_primary_associations,
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

In [41]:
# =============================================================================
# Compare pooled and lineage-adjusted effect directions
# =============================================================================

primary_significant_evidence["pooled_adjusted_direction_consistent"] = (
    np.sign(primary_significant_evidence["pooled_spearman_rho"])
    == np.sign(primary_significant_evidence["beta"])
)

pooled_adjusted_direction_summary = (
    primary_significant_evidence
    .groupby("consensus_program_id")
    .agg(
        n_associations=("source_gene_label", "size"),
        n_direction_consistent=(
            "pooled_adjusted_direction_consistent",
            "sum",
        ),
        direction_consistency_fraction=(
            "pooled_adjusted_direction_consistent",
            "mean",
        ),
    )
    .reset_index()
)

pooled_adjusted_direction_summary

,consensus_program_id,n_associations,n_direction_consistent,direction_consistency_fraction
0,CONSENSUS_TX_01,307,303,0.986971
1,CONSENSUS_TX_02,253,253,1.000000


In [42]:
# =============================================================================
# Identify lineage-sensitive direction reversals
# =============================================================================

lineage_sensitive_direction_reversals = (
    primary_significant_evidence.loc[
        ~primary_significant_evidence[
            "pooled_adjusted_direction_consistent"
        ],
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "beta",
            "fdr_q_value",
            "pooled_spearman_rho",
            "direction_consistency_fraction",
            "lineage_beta_sd",
        ],
    ]
    .copy()
)

lineage_sensitive_direction_reversals

,consensus_program_id,source_gene_label,gene_symbol,beta,fdr_q_value,pooled_spearman_rho,direction_consistency_fraction,lineage_beta_sd
11,CONSENSUS_TX_01,AMBN (258),AMBN,-0.056906,0.029131,0.015984,0.692308,0.077020
29,CONSENSUS_TX_01,CEPT1 (10390),CEPT1,-0.182683,0.006762,0.015974,0.615385,0.414070
131,CONSENSUS_TX_01,MEF2B (100271849),MEF2B,-0.211763,0.000507,0.015406,0.461538,0.160326
240,CONSENSUS_TX_01,RIPPLY2 (134701),RIPPLY2,0.042935,0.040533,-0.035028,0.615385,0.078583


In [43]:
# =============================================================================
# Construct lineage-specific profiles for direction reversals
# =============================================================================

lineage_sensitive_profiles = (
    within_lineage_associations
    .merge(
        lineage_sensitive_direction_reversals[
            [
                "consensus_program_id",
                "source_gene_label",
                "gene_symbol",
                "beta",
                "pooled_spearman_rho",
            ]
        ].rename(columns={"beta": "primary_beta"}),
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="inner",
        validate="many_to_one",
    )
)

lineage_sensitive_profiles["direction_consistent_with_primary"] = (
    np.sign(lineage_sensitive_profiles["beta"])
    == np.sign(lineage_sensitive_profiles["primary_beta"])
)

In [44]:
# =============================================================================
# Summarize lineage contributions to direction reversals
# =============================================================================

lineage_sensitive_profiles["primary_opposing_lineage"] = (
    lineage_sensitive_profiles["OncotreeLineage"].where(
        ~lineage_sensitive_profiles[
            "direction_consistent_with_primary"
        ]
    )
)

lineage_sensitive_summary = (
    lineage_sensitive_profiles
    .groupby(
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
        ]
    )
    .agg(
        primary_beta=("primary_beta", "first"),
        pooled_spearman_rho=("pooled_spearman_rho", "first"),
        n_evaluable_lineages=("OncotreeLineage", "nunique"),
        n_direction_consistent=(
            "direction_consistent_with_primary",
            "sum",
        ),
        n_direction_opposed=(
            "direction_consistent_with_primary",
            lambda x: (~x).sum(),
        ),
        opposing_lineages=(
            "primary_opposing_lineage",
            lambda x: "; ".join(x.dropna()),
        ),
    )
    .reset_index()
)

lineage_sensitive_summary

,consensus_program_id,source_gene_label,gene_symbol,primary_beta,pooled_spearman_rho,n_evaluable_lineages,n_direction_consistent,n_direction_opposed,opposing_lineages
0,CONSENSUS_TX_01,AMBN (258),AMBN,-0.056906,0.015984,13,9,4,CNS/Brain; Pancreas; Peripheral Nervous System...
1,CONSENSUS_TX_01,CEPT1 (10390),CEPT1,-0.182683,0.015974,13,8,5,Esophagus/Stomach; Bowel; Breast; Pancreas; Skin
2,CONSENSUS_TX_01,MEF2B (100271849),MEF2B,-0.211763,0.015406,13,6,7,Lung; Esophagus/Stomach; CNS/Brain; Breast; He...
3,CONSENSUS_TX_01,RIPPLY2 (134701),RIPPLY2,0.042935,-0.035028,13,8,5,Ovary/Fallopian Tube; Pancreas; Skin; Peripher...


## Program overlap and sensitivity to lineage adjustment

**Saved execution.** The 560 significant associations cover 532 unique genes. Of the 28 genes associated with both programs, 27 had the same effect direction across programs and one had opposite directions.

Pooled and lineage-adjusted directions agreed for 556 of 560 associations. All four reversals involved `CONSENSUS_TX_01` (`AMBN`, `CEPT1`, `MEF2B`, and `RIPPLY2`).

In [45]:
# =============================================================================
# Construct integrated primary CRISPR evidence table
# =============================================================================

primary_significant_evidence["n_significant_programs_for_gene"] = (
    primary_significant_evidence["source_gene_label"]
    .map(significant_gene_program_count)
)

primary_significant_evidence["lineage_sensitive_direction_reversal"] = (
    ~primary_significant_evidence[
        "pooled_adjusted_direction_consistent"
    ]
)

crispr_primary_evidence_table = (
    primary_significant_evidence[
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "n_models",
            "beta",
            "standard_error",
            "p_value",
            "fdr_q_value",
            "effect_direction",
            "absolute_beta",
            "n_evaluable_lineages",
            "n_direction_consistent",
            "direction_consistency_fraction",
            "median_lineage_beta",
            "lineage_beta_sd",
            "lineage_beta_iqr",
            "lineage_beta_range",
            "pooled_spearman_rho",
            "pooled_adjusted_direction_consistent",
            "lineage_sensitive_direction_reversal",
            "n_significant_programs_for_gene",
        ]
    ]
    .copy()
)

In [46]:
# =============================================================================
# Define putative vulnerability association subset
# =============================================================================

crispr_putative_vulnerability_associations = (
    crispr_primary_evidence_table.loc[
        crispr_primary_evidence_table["effect_direction"]
        == "higher_program_stronger_dependency"
    ]
    .copy()
)

In [47]:
# =============================================================================
# Characterize putative vulnerability gene overlap
# =============================================================================

putative_vulnerability_gene_program_count = (
    crispr_putative_vulnerability_associations.groupby("source_gene_label")[
        "consensus_program_id"
    ].nunique()
)

print(
    "Putative vulnerability associations:",
    len(crispr_putative_vulnerability_associations),
)

print(
    "Unique putative vulnerability genes:",
    putative_vulnerability_gene_program_count.size,
)

for n_programs in range(1, len(consensus_program_ids) + 1):
    print(
        f"Genes associated with exactly {n_programs} program(s):",
        putative_vulnerability_gene_program_count.eq(n_programs).sum(),
    )

Putative vulnerability associations: 341
Unique putative vulnerability genes: 327
Genes associated with one program: 313
Genes associated with two programs: 14


In [48]:
# =============================================================================
# Summarize putative vulnerability evidence by program
# =============================================================================

putative_vulnerability_summary = (
    crispr_putative_vulnerability_associations
    .groupby("consensus_program_id")
    .agg(
        n_associations=("source_gene_label", "size"),
        n_unique_genes=("source_gene_label", "nunique"),
        median_beta=("beta", "median"),
        median_absolute_beta=("absolute_beta", "median"),
        median_direction_consistency=(
            "direction_consistency_fraction",
            "median",
        ),
        median_lineage_beta_sd=("lineage_beta_sd", "median"),
        median_lineage_beta_iqr=("lineage_beta_iqr", "median"),
        median_lineage_beta_range=("lineage_beta_range", "median"),
    )
    .reset_index()
)

putative_vulnerability_summary

,consensus_program_id,n_associations,n_unique_genes,median_beta,median_absolute_beta,median_direction_consistency,median_lineage_beta_sd,median_lineage_beta_iqr,median_lineage_beta_range
0,CONSENSUS_TX_01,226,226,-0.144598,0.144598,0.807692,0.221930,0.229756,0.772690
1,CONSENSUS_TX_02,115,115,-0.049984,0.049984,0.692308,0.137987,0.091832,0.501501


## Putative vulnerability associations

**Saved execution.** The significant associations with negative coefficients comprise 341 program–gene pairs across 327 unique genes. `CONSENSUS_TX_01` contributes 226 pairs and `CONSENSUS_TX_02` contributes 115. Fourteen genes show the stronger-dependency direction in both programs.

In [49]:
# =============================================================================
# Freeze descriptive ordering for gene-level interpretation
# =============================================================================

crispr_putative_vulnerability_table = (
    crispr_putative_vulnerability_associations
    .sort_values(
        by=[
            "fdr_q_value",
            "beta",
            "source_gene_label",
        ],
        ascending=[
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

In [50]:
# =============================================================================
# Construct descriptive putative vulnerability evidence table
# =============================================================================

crispr_putative_vulnerability_table = (
    crispr_putative_vulnerability_table[
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "n_models",
            "beta",
            "standard_error",
            "p_value",
            "fdr_q_value",
            "absolute_beta",
            "n_evaluable_lineages",
            "n_direction_consistent",
            "direction_consistency_fraction",
            "median_lineage_beta",
            "lineage_beta_sd",
            "lineage_beta_iqr",
            "lineage_beta_range",
            "pooled_spearman_rho",
            "lineage_sensitive_direction_reversal",
            "n_significant_programs_for_gene",
        ]
    ]
    .copy()
)

In [51]:
# =============================================================================
# Preview descriptively ordered putative vulnerability associations
# =============================================================================

PUTATIVE_VULNERABILITY_PREVIEW_N = 20

crispr_putative_vulnerability_table.head(
    PUTATIVE_VULNERABILITY_PREVIEW_N
)

,consensus_program_id,source_gene_label,gene_symbol,entrez_id,n_models,beta,standard_error,p_value,fdr_q_value,absolute_beta,n_evaluable_lineages,n_direction_consistent,direction_consistency_fraction,median_lineage_beta,lineage_beta_sd,lineage_beta_iqr,lineage_beta_range,pooled_spearman_rho,lineage_sensitive_direction_reversal,n_significant_programs_for_gene
0,CONSENSUS_TX_02,CDS2 (8760),CDS2,8760,537,-0.104399,0.013922,6.443436e-14,5.406047e-10,0.104399,13,9,0.692308,-0.054231,0.277757,0.210299,1.054340,-0.485690,False,2
1,CONSENSUS_TX_01,EBF1 (1879),EBF1,1879,537,-0.337719,0.049442,8.453712e-12,3.636153e-08,0.337719,13,9,0.692308,-0.032214,0.188844,0.120172,0.763991,-0.260212,False,1
2,CONSENSUS_TX_02,ELMO2 (63916),ELMO2,63916,537,-0.111760,0.017315,1.085495e-10,2.873222e-07,0.111760,13,8,0.615385,-0.057995,0.196583,0.147080,0.732144,-0.350909,False,1
3,CONSENSUS_TX_02,PSMB5 (5693),PSMB5,5693,537,-0.209848,0.034152,8.022074e-10,1.971711e-06,0.209848,13,12,0.923077,-0.417989,0.669226,0.821761,2.376894,-0.160364,False,1
4,CONSENSUS_TX_01,MRPS10 (55173),MRPS10,55173,537,-0.183215,0.033038,2.929613e-08,5.929882e-05,0.183215,13,13,1.000000,-0.266590,0.176905,0.174265,0.585128,-0.285860,False,1
5,CONSENSUS_TX_02,ITGAV (3685),ITGAV,3685,537,-0.138980,0.026060,9.655585e-08,1.328995e-04,0.138980,13,11,0.846154,-0.093829,0.158799,0.150939,0.625211,-0.312439,False,1
6,CONSENSUS_TX_01,CDS2 (8760),CDS2,8760,537,-0.213122,0.040665,1.597191e-07,2.071446e-04,0.213122,13,8,0.615385,-0.259948,0.391395,0.456442,1.515130,-0.351638,False,2
7,CONSENSUS_TX_01,SLC25A51 (92014),SLC25A51,92014,532,-0.121145,0.023129,1.625371e-07,2.071446e-04,0.121145,13,11,0.846154,-0.196798,0.179580,0.191499,0.671303,-0.251372,False,1
8,CONSENSUS_TX_01,FARS2 (10667),FARS2,10667,537,-0.260861,0.050212,2.045408e-07,2.426983e-04,0.260861,13,12,0.923077,-0.337915,0.296452,0.319356,1.078109,-0.336075,False,1
9,CONSENSUS_TX_01,WDR37 (22884),WDR37,22884,537,-0.099499,0.019386,2.858281e-07,2.892748e-04,0.099499,13,11,0.846154,-0.079629,0.100067,0.121764,0.328469,-0.357677,False,1


In [52]:
# =============================================================================
# Output directories
# =============================================================================

DEPENDENCY_OUTPUT_DIR = Paths.dependencies
FUNCTIONAL_VULNERABILITY_OUTPUT_DIR = (
    Paths.functional_vulnerabilities
)

In [53]:
# =============================================================================
# Output artifact paths
# =============================================================================

CRISPR_MODEL_COHORT_PATH = (
    DEPENDENCY_OUTPUT_DIR
    / "500_crispr_model_cohort.csv"
)

CRISPR_GENE_COVERAGE_PATH = (
    DEPENDENCY_OUTPUT_DIR
    / "500_crispr_gene_coverage.csv"
)

CRISPR_PRIMARY_ASSOCIATIONS_PATH = (
    FUNCTIONAL_VULNERABILITY_OUTPUT_DIR
    / "500_crispr_primary_associations.csv"
)

CRISPR_SIGNIFICANT_EVIDENCE_PATH = (
    FUNCTIONAL_VULNERABILITY_OUTPUT_DIR
    / "500_crispr_primary_significant_evidence.csv"
)

CRISPR_WITHIN_LINEAGE_PATH = (
    FUNCTIONAL_VULNERABILITY_OUTPUT_DIR
    / "500_crispr_within_lineage_associations.csv"
)

CRISPR_PUTATIVE_VULNERABILITY_PATH = (
    FUNCTIONAL_VULNERABILITY_OUTPUT_DIR
    / "500_crispr_putative_vulnerability_associations.csv"
)

CRISPR_METADATA_PATH = (
    FUNCTIONAL_VULNERABILITY_OUTPUT_DIR
    / "500_crispr_analysis_metadata.json"
)

In [54]:
# =============================================================================
# Construct CRISPR shared-model cohort artifact
# =============================================================================

crispr_model_cohort = (
    shared_consensus_analysis
    .reset_index()
    [
        [
            "ModelID",
            "OncotreeLineage",
            *consensus_program_ids,
        ]
    ]
    .copy()
)

In [55]:
# =============================================================================
# Construct CRISPR gene-coverage artifact
# =============================================================================

crispr_gene_coverage_artifact = (
    crispr_gene_coverage[
        [
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "n_observed",
            "n_missing",
            "missing_fraction",
            "gene_effect_sd",
            "n_unique",
            "analysis_eligible",
        ]
    ]
    .copy()
)

In [56]:
# =============================================================================
# Construct primary CRISPR association artifact
# =============================================================================

crispr_primary_association_artifact = (
    primary_crispr_associations
    .merge(
        crispr_gene_annotations[
            [
                "source_gene_label",
                "gene_symbol",
                "entrez_id",
            ]
        ],
        on="source_gene_label",
        how="left",
        validate="many_to_one",
    )
    [
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "n_models",
            "beta",
            "standard_error",
            "p_value",
            "fdr_q_value",
            "fdr_significant",
        ]
    ]
    .copy()
)

In [57]:
# =============================================================================
# Construct significant CRISPR evidence artifact
# =============================================================================

crispr_significant_evidence_artifact = (
    crispr_primary_evidence_table
    .copy()
)

In [58]:
# =============================================================================
# Construct within-lineage CRISPR association artifact
# =============================================================================

crispr_within_lineage_artifact = (
    within_lineage_associations
    .merge(
        crispr_gene_annotations[
            [
                "source_gene_label",
                "gene_symbol",
                "entrez_id",
            ]
        ],
        on="source_gene_label",
        how="left",
        validate="many_to_one",
    )
    [
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "OncotreeLineage",
            "n_models",
            "beta",
        ]
    ]
    .copy()
)

In [59]:
# =============================================================================
# Construct putative vulnerability CRISPR artifact
# =============================================================================

crispr_putative_vulnerability_artifact = (
    crispr_putative_vulnerability_table
    .copy()
)

In [60]:
# =============================================================================
# Construct CRISPR analysis metadata
# =============================================================================

crispr_analysis_metadata = {
    "notebook": "500_crispr_associations",
    "source_dataset": "DepMap Public 24Q4",
    "source_file": "CRISPRGeneEffect.csv",
    "frozen_consensus_programs": consensus_program_ids,
    "frozen_consensus_model_count": len(consensus_cellline_scores),
    "crispr_source_model_count": len(crispr_gene_effect),
    "shared_model_count": len(shared_consensus_analysis),
    "observed_lineage_count": len(shared_lineage_counts),
    "primary_min_lineage_n": PRIMARY_MIN_LINEAGE_N,
    "secondary_min_lineage_n": MIN_LINEAGE_N,
    "supported_lineage_count": len(supported_lineages),
    "crispr_source_gene_count": len(crispr_gene_annotations),
    "min_crispr_model_fraction": MIN_CRISPR_MODEL_FRACTION,
    "min_crispr_models": MIN_CRISPR_MODELS,
    "eligible_crispr_gene_count": len(eligible_crispr_genes),
    "primary_hypothesis_count": PRIMARY_HYPOTHESIS_COUNT,
    "primary_model": "gene_effect ~ consensus_score + lineage_fixed_effects",
    "primary_covariance": PRIMARY_COV_TYPE,
    "multiple_testing_method": PRIMARY_FDR_METHOD,
    "primary_fdr_alpha": PRIMARY_FDR_ALPHA,
    "primary_fdr_significant_count": int(
        primary_crispr_associations["fdr_significant"].sum()
    ),
    "putative_vulnerability_association_count": len(
        crispr_putative_vulnerability_artifact
    ),
    "secondary_characterization": [
        "within_lineage_directional_consistency",
        "cross_lineage_effect_heterogeneity",
        "pooled_spearman_context",
        "multi_program_gene_overlap",
    ],
    "interpretation_boundary": (
        "Computational dependency associations and putative functional "
        "vulnerability evidence; not causal dependency, target validation, "
        "therapeutic efficacy, clinical prediction, or independent replication."
    ),
}

In [61]:
# =============================================================================
# Write analysis-ready CRISPR dependency artifacts
# =============================================================================

crispr_model_cohort.to_csv(
    CRISPR_MODEL_COHORT_PATH,
    index=False,
)

crispr_gene_coverage_artifact.to_csv(
    CRISPR_GENE_COVERAGE_PATH,
    index=False,
)

In [62]:
# =============================================================================
# Write processed CRISPR association artifacts
# =============================================================================

crispr_primary_association_artifact.to_csv(
    CRISPR_PRIMARY_ASSOCIATIONS_PATH,
    index=False,
)

crispr_significant_evidence_artifact.to_csv(
    CRISPR_SIGNIFICANT_EVIDENCE_PATH,
    index=False,
)

crispr_within_lineage_artifact.to_csv(
    CRISPR_WITHIN_LINEAGE_PATH,
    index=False,
)

crispr_putative_vulnerability_artifact.to_csv(
    CRISPR_PUTATIVE_VULNERABILITY_PATH,
    index=False,
)

In [63]:
# =============================================================================
# Write CRISPR analysis metadata
# =============================================================================

with CRISPR_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        crispr_analysis_metadata,
        metadata_file,
        indent=2,
    )

In [65]:
# =============================================================================
# Verify written notebook 500 artifacts
# =============================================================================

import csv
from io import StringIO
from itertools import zip_longest

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    refresh_notebook_artifacts,
)

csv_artifacts = {
    CRISPR_MODEL_COHORT_PATH: crispr_model_cohort,
    CRISPR_GENE_COVERAGE_PATH: crispr_gene_coverage_artifact,
    CRISPR_PRIMARY_ASSOCIATIONS_PATH: crispr_primary_association_artifact,
    CRISPR_SIGNIFICANT_EVIDENCE_PATH: crispr_significant_evidence_artifact,
    CRISPR_WITHIN_LINEAGE_PATH: crispr_within_lineage_artifact,
    CRISPR_PUTATIVE_VULNERABILITY_PATH: crispr_putative_vulnerability_artifact,
}

verified_outputs = {}
# Compare parsed CSV fields against the intended serialization. This preserves
# identifier spelling and numeric precision while allowing CSV quoting differences.
for artifact_path, artifact_table in csv_artifacts.items():
    expected_csv = StringIO(artifact_table.to_csv(index=False), newline="")
    with artifact_path.open(encoding="utf-8", newline="") as handle:
        persisted = csv.reader(handle, strict=True)
        expected = csv.reader(expected_csv, strict=True)
        for row_number, (row, expected_row) in enumerate(
            zip_longest(persisted, expected)
        ):
            if row != expected_row:
                location = "header" if row_number == 0 else f"data row {row_number}"
                raise ValueError(f"{artifact_path.name}: persisted {location} differs")
    verified_outputs[artifact_path] = (row_number, len(artifact_table.columns))

with CRISPR_METADATA_PATH.open("r", encoding="utf-8") as metadata_file:
    written_metadata = json.load(metadata_file)
if written_metadata != crispr_analysis_metadata:
    raise ValueError(f"Metadata mismatch for {CRISPR_METADATA_PATH.name}")
verified_outputs[CRISPR_METADATA_PATH] = None

notebook_path = (
    "notebooks/phase5_functional_vulnerabilities/" "500_crispr_associations.ipynb"
)
updated_count = refresh_notebook_artifacts(
    notebook_path, verified_outputs, require_unchanged=False
)

registry = load_artifact_registry()
registered_shapes = {
    entry["path"]: entry["shape"]
    for entry in registry["artifacts"].values()
    if entry["producer"].get("path") == notebook_path
}
verified_shapes = {
    project_relative_path(path): list(shape) if shape is not None else None
    for path, shape in verified_outputs.items()
}
if registered_shapes != verified_shapes:
    raise ValueError(f"Refreshed registry shapes differ for {notebook_path}")

print("Registry entries refreshed:", updated_count)

CSV artifacts verified: 6 of 6
Metadata verified: True
Notebook 500 artifact verification: True
Registry entries refreshed: 7


## Conclusion

The lineage-adjusted CRISPR screen identifies program-specific associations consistent with stronger dependency. These are hypothesis-generating findings. Residual confounding, including proliferation-related effects, remains possible.

Notebook 501 characterizes RNAi evidence independently. Notebook 502 integrates the two platform-specific association layers.